In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40) # makes it so columns not hidden
pd.set_option("display.width", 140) # wider console output
pd.set_option("display.float_format", lambda x: f"{x:,.4f}") # thousands + 4 decimals

print("pandas :", pd.__version__)
print("numpy  :", np.__version__)

pandas : 2.3.3
numpy  : 2.3.5


In [5]:
data_path = "v23_2_dataset_vx_fcst.xlsx"

import os
assert os.path.exists(data_path), (
    f"Could not find '{data_path}'. Make sure the .xlsx is in the same folder as this notebook, "
    f"or edit data_path to the full path."
)
print("Found file:", data_path, f"({os.path.getsize(data_path)/1024:.0f} KB)")

Found file: v23_2_dataset_vx_fcst.xlsx (494 KB)


# 1.1 Reading messy Excel *properly*

Beginners often do `pd.read_excel(path)` and immediately start analyzing whatever comes back.
That's a trap. Real business spreadsheets have **title banners, notes, and blank rows** above the
actual table. If you don't tell pandas where the real header row is, every column name and every
dtype will be wrong — and you won't necessarily get an error, just silently bad data.

**So step one is always: look at the raw sheet before parsing it.**

## 1.1 What sheets are in the workbook?

`pd.ExcelFile` opens the workbook *once* and lets us inspect it cheaply before reading anything heavy.

In [7]:
xl = pd.ExcelFile(data_path)
xl.sheet_names

['v23.2 Forecast Data (Internal)', 'v23.2 Key Prog Assumptions', 'Pivot']

Three sheets:
- v23.2 Forecast Data (Internal) -- the main cost/volume table.
- v23.2 Key Prog Assumptions -- the coverage/population/wastage drivers.
- Pivot -- a pre-built Excel summary we'll reproduce as a self-check.

## 1.2 Peek at the raw top-left corner (no header assumptions)

We read with header=None so pandas makes no assumption about where the header is.  Then we look at the first several rows and columns to *see* the layout with our own eyes.

In [8]:
raw = pd.read_excel(data_path, sheet_name="v23.2 Forecast Data (Internal)", header=None)
print("Raw shape (rows, cols):",raw.shape)
raw.iloc[:8, :8] # first 8 rows, first 8 columns

Raw shape (rows, cols): (2347, 13)


,0,1,2,3,4,5,6,7
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,v23.2 Dataset - Vaccine Procurement Cost & Vol...,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,2026-2030 Forecast by Antigen and Support Type...,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Please note that the official SAC dataset as w...,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,AFC LVL1,Support Type,Antigen,Year,Fund Center,Forecast,Period
6,NaN,Vaccine,Campaign,HPV,2029,Afghanistan,v23.2,Gavi 6.0
7,NaN,Vaccine,Routine,MR,2029,Afghanistan,v23.2,Gavi 6.0
